In [6]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
import deeplake as dl
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA

In [ ]:
# Loading the Dataset
train_ds = dl.load("hub://activeloop/visdrone-det-train")
val_ds = dl.load("hub://activeloop/visdrone-det-val")
test_ds = dl.load("hub://activeloop/visdrone-det-test-dev")

In [ ]:
# Global Variables and Configs
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

# Image Constants
IMG_HEIGHT = 224
IMG_WIDTH = 407
IMG_CHANNELS = 3
IMAGE_SHAPE = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)

GLOBAL_BACKBONE = keras.applications.MobileNetV2(
    weights='imagenet', 
    include_top=False,
    input_shape=IMAGE_SHAPE
)
GLOBAL_BACKBONE.trainable = False

city_labels = ["tianjin", "hong-kong", "daqing", "ganzhou", 
               "guangzhou", "jinchang", "liuzhou", "nanjing", 
               "shaoxing", "shenyang", "nanyang", "zhangjiakou", 
               "suzhou", "xuzhou"]

In [4]:
# Preprocessing Detection

def image_preprocessing(image):
    image = tf.image.resize(image, (IMG_HEIGHT, IMG_WIDTH))
    image = keras.applications.mobilenet_v2.preprocess_input(image)
    return image

def detection_preprocessing(data):
    image = image_preprocessing(data['images'])
    labels = data['labels']  # Array of integer Class IDs
    
    # 1. Track Humans: Pedestrians (1) OR People (2)
    is_human = tf.reduce_any(tf.equal(tf.expand_dims(labels, -1), [1, 2]), axis=-1)
    human_count = tf.reduce_sum(tf.cast(is_human, tf.float32))
    
    # 2. Track Vehicles: Car (4), Vans (5), Trucks (6), Buses (9), Motorcycles (10)
    is_vehicle = tf.reduce_any(tf.equal(tf.expand_dims(labels, -1), [4, 5, 6, 9, 10]), axis=-1)
    vehicle_count = tf.reduce_sum(tf.cast(is_vehicle, tf.float32))
    
    counts_vector = tf.stack([human_count, vehicle_count], axis=0)
    counts_vector = tf.math.log1p(counts_vector)
    
    return image, counts_vector

# Feature Stream Pipelines

train_stream = (train_ds.tensorflow()
                 .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                 .batch(BATCH_SIZE)
                 .prefetch(AUTOTUNE))
val_stream = (val_ds.tensorflow()
                .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                .batch(BATCH_SIZE)
                .prefetch(AUTOTUNE))
test_stream = (test_ds.tensorflow()
                 .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                 .batch(BATCH_SIZE)
                 .prefetch(AUTOTUNE))

In [8]:
# Preprocessing City Labels

@tf.function
def forward_pass(images):
    feature_maps = GLOBAL_BACKBONE(images, training=False)
    return tf.reduce_mean(feature_maps, axis=[1, 2])

def extract_features(dataset_stream):
    feature_chunks = []
    for images, _ in dataset_stream:
        feats = forward_pass(images)
        feature_chunks.append(feats)

    return tf.concat(feature_chunks, axis=0) 

train_features = extract_features(train_stream).numpy()
val_features = extract_features(val_stream).numpy()
test_features = extract_features(test_stream).numpy()

trained_pca = PCA(n_components=50, random_state=42)
trained_kmeans = MiniBatchKMeans(n_clusters=14, random_state=42, n_init=5, batch_size=1024)

tf_train_labels = trained_kmeans.fit_predict(trained_pca.fit_transform(train_features))
tf_val_labels = trained_kmeans.predict(trained_pca.transform(val_features))
tf_test_labels = trained_kmeans.predict(trained_pca.transform(test_features))

train_labels_ds = tf.data.Dataset.from_tensor_slices(tf_train_labels).batch(BATCH_SIZE)
val_labels_ds = tf.data.Dataset.from_tensor_slices(tf_val_labels).batch(BATCH_SIZE)
test_labels_ds = tf.data.Dataset.from_tensor_slices(tf_test_labels).batch(BATCH_SIZE)

In [9]:
# Adding Pipelines

def structure_outputs(img_batch, count_batch, label_batch):
    img_batch.set_shape([None, *IMAGE_SHAPE])
    count_batch.set_shape([None, 2])
    label_batch.set_shape([None])
    
    return img_batch, {
        "city_output": label_batch,
        "count_output": count_batch
    }

# Zip your feature streams (images + counts) with your clustering labels
train_pipeline = tf.data.Dataset.zip((train_stream, train_labels_ds)).map(
    lambda stream_data, labels: structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

val_pipeline = tf.data.Dataset.zip((val_stream, val_labels_ds)).map(
    lambda stream_data, labels: structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

test_pipeline = tf.data.Dataset.zip((test_stream, test_labels_ds)).map(
    lambda stream_data, labels: structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

In [10]:
# Building the Model
def build_multi_head_model():
    input_tensor = layers.Input(shape=(IMAGE_SHAPE), name='input_image')

    shared_features = GLOBAL_BACKBONE(input_tensor)

    # 1. Object Detection Head (Processes backbone features first)
    x_detect = layers.Conv2D(64, 3, padding="same", activation='relu')(shared_features)
    x_detect = layers.Conv2D(32, 3, padding="same", activation='relu')(x_detect)
    x_detect = layers.GlobalAveragePooling2D()(x_detect)
    
    x_detect = layers.Dense(128, activation=None, kernel_initializer='he_normal')(x_detect)
    x_detect = layers.BatchNormalization()(x_detect)
    x_detect = layers.LeakyReLU(negative_slope=0.1)(x_detect)
    
    # Final count output layer
    count_output = layers.Dense(2, activation='linear', kernel_initializer='he_normal', name="count_output")(x_detect)

    # 2. City Classification Head (Dependent on Detection)
    
    protected_counts = layers.Lambda(lambda x: tf.stop_gradient(x))(count_output)

    x_class_input = layers.GlobalAveragePooling2D()(shared_features)
    
    # Concatenate blends the spatial context with the localization features
    merged_features = layers.Concatenate()([x_class_input, protected_counts])
    
    x_class = layers.Dense(256, activation='relu')(merged_features)
    x_class = layers.Dropout(0.5)(x_class)
    city_output = layers.Dense(14, activation='softmax', name="city_output")(x_class)

    model = keras.Model(inputs=input_tensor, outputs=[city_output, count_output])

    lr_scheduler = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.001,
        decay_steps=10000,
        alpha=0.0
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_scheduler),
        loss={
            "city_output": "sparse_categorical_crossentropy",
            "count_output": "huber"
        },
        metrics={
            "city_output": "accuracy",
            "count_output": "mae"
        },
        loss_weights={
            "city_output": 0.5, 
            "count_output": 1.0
        }
    )
    return model

tf.keras.backend.clear_session() # Clearing for next session.
china_model = build_multi_head_model()

history = china_model.fit(train_pipeline, validation_data=val_pipeline, epochs=3)

print(china_model.summary())


Epoch 1/3
    405/Unknown 264s 640ms/step - city_output_accuracy: 0.5345 - city_output_loss: 1.3969 - count_output_loss: 0.8333 - count_output_mae: 1.2383 - loss: 1.5317

c:\Users\etito\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


405/405 ━━━━━━━━━━━━━━━━━━━━ 284s 691ms/step - city_output_accuracy: 0.6529 - city_output_loss: 0.9792 - count_output_loss: 0.5188 - count_output_mae: 0.8997 - loss: 1.0112 - val_city_output_accuracy: 0.8431 - val_city_output_loss: 0.4559 - val_count_output_loss: 0.2762 - val_count_output_mae: 0.6129 - val_loss: 0.5163
Epoch 2/3
405/405 ━━━━━━━━━━━━━━━━━━━━ 275s 678ms/step - city_output_accuracy: 0.7901 - city_output_loss: 0.5580 - count_output_loss: 0.3072 - count_output_mae: 0.6499 - loss: 0.5875 - val_city_output_accuracy: 0.8686 - val_city_output_loss: 0.3424 - val_count_output_loss: 0.2535 - val_count_output_mae: 0.5938 - val_loss: 0.4385
Epoch 3/3
405/405 ━━━━━━━━━━━━━━━━━━━━ 277s 684ms/step - city_output_accuracy: 0.8220 - city_output_loss: 0.4774 - count_output_loss: 0.2519 - count_output_mae: 0.5742 - loss: 0.4920 - val_city_output_accuracy: 0.8449 - val_city_output_loss: 0.3600 - val_count_output_loss: 0.2659 - val_count_output_mae: 0.6165 - val_loss: 0.4625


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 224, 407,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_2… │ (None, 7, 13,     │  2,257,984 │ input_image[0][0] │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 7, 13, 64) │    737,344 │ mobilenetv2_1.00… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 7, 13, 32) │     18,464 │ conv2d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ conv2d_1[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │      4,224 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128)       │        512 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu         │ (None, 128)       │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ count_output        │ (None, 2)         │        258 │ leaky_re_lu[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 2)         │          0 │ count_output[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1282)      │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │    328,448 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ city_output (Dense) │ (None, 14)        │      3,598 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,536,017 (21.12 MB)

 Trainable params: 1,092,592 (4.17 MB)

 Non-trainable params: 2,258,240 (8.61 MB)

 Optimizer params: 2,185,185 (8.34 MB)

None


In [16]:
# Testing Model

results = china_model.evaluate(test_pipeline, return_dict=True)

for test_images, test_targets in test_pipeline.take(1):
    city_pred, count_pred = china_model.predict(test_images, verbose=0)

    human_counts = np.expm1(count_pred[:, 0])
    vehicle_counts = np.expm1(count_pred[:, 1])
    
    predicted_city_indices = np.argmax(city_pred, axis=-1)
    true_city_indices = np.array(test_targets["city_output"]).astype(int).flatten()
    true_counts = np.array(test_targets["count_output"])
    
    for i in range(5):
        pred_label = city_labels[predicted_city_indices[i]]
        true_label = city_labels[true_city_indices[i]]
        
        print(f"\nSample {i+1}:")
        print(f"[City] Predicted: {pred_label:<12} | True: {true_label}")
        print(f"[Human] Predicted: {count_pred[i][0]:<12.2f} | True: {true_counts[i][0]:.1f}")
        print(f"[Car] Predicted: {count_pred[i][1]:<12.2f} | True: {true_counts[i][1]:.1f}")

101/101 ━━━━━━━━━━━━━━━━━━━━ 57s 564ms/step - city_output_accuracy: 0.8453 - city_output_loss: 0.3882 - count_output_loss: 0.4437 - count_output_mae: 0.8180 - loss: 0.6444

Sample 1:
[City] Predicted: nanyang      | True: nanyang
[Human] Predicted: 0.81         | True: 0.0
[Car] Predicted: 2.03         | True: 2.8

Sample 2:
[City] Predicted: jinchang     | True: jinchang
[Human] Predicted: 1.87         | True: 0.0
[Car] Predicted: 4.04         | True: 3.4

Sample 3:
[City] Predicted: jinchang     | True: jinchang
[Human] Predicted: 2.39         | True: 4.0
[Car] Predicted: 3.49         | True: 3.9

Sample 4:
[City] Predicted: suzhou       | True: suzhou
[Human] Predicted: 2.97         | True: 2.6
[Car] Predicted: 4.10         | True: 3.5

Sample 5:
[City] Predicted: daqing       | True: daqing
[Human] Predicted: 1.79         | True: 0.0
[Car] Predicted: 2.93         | True: 3.6
